# Qwen3-TTS VoiceDesign - Khaya Wants to Row

Generate original synthetic reference voices for Narrator, Khaya, Gogo, and Mkhulu on a Colab L4 GPU. Outputs are written to `/content/qwen_voice_design` and, when Drive is mounted, copied to `MyDrive/vidio/khaya-wants-to-row/voices`.

In [ ]:
# @title 1. Verify GPU and install dependencies
import subprocess, sys
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'qwen-tts', 'soundfile'])
print('Dependencies installed.')

In [ ]:
# @title 2. Load Qwen3-TTS VoiceDesign
import json, os, random
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
from qwen_tts import Qwen3TTSModel

MODEL_ID = 'Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign'
OUT_DIR = Path('/content/qwen_voice_design')
OUT_DIR.mkdir(parents=True, exist_ok=True)

model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map='cuda:0',
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
)
print('Loaded', MODEL_ID)

In [ ]:
# @title 3. Character voice specifications
VOICE_SPECS = {
    'narrator': {
        'text': ('On a bright morning, a small boat waited beside the quiet river. '
                 'The water moved gently, and a new adventure was about to begin.'),
        'instruct': ('A warm adult female storybook narrator speaking clear English. Gentle, reassuring and expressive, '
                     'with a subtle South African English character. Medium-low pitch, unhurried pace, a soft smile in '
                     'the voice, and excellent articulation. Natural and intimate, never theatrical, breathy or overly '
                     'emotional. Clean studio recording with no music, noise or reverberation.')
    },
    'khaya': {
        'text': ('Look at the boats! They are moving so fast. One day, I will learn to row across the water too.'),
        'instruct': ('A wholly synthetic fictional young boy storybook voice, approximately six to eight years old. '
                     'Bright, curious, energetic and sincere, speaking clear English with a subtle South African English '
                     'character. Medium-high pitch but not shrill, with a natural childlike rhythm. Lively without shouting '
                     'or exaggerated cartoon acting. Not an imitation of any real child. Clean studio recording with no '
                     'music, noise or reverberation.')
    },
    'gogo': {
        'text': ('Be patient, little one. Dreams grow stronger when you care for them every day. '
                 'Keep believing, and you will find your way.'),
        'instruct': ('A warm elderly woman storybook voice speaking clear English. Gentle, wise, affectionate and reassuring, '
                     'with a subtle South African English character. Medium pitch, relaxed pace, a soft smile and natural age '
                     'texture. Clear articulation without sounding weak, breathy or frail. Clean studio recording with no '
                     'music, noise or reverberation.')
    },
    'mkhulu': {
        'text': ('Hold the oar gently, watch the water, and take your time. You will learn, step by step, '
                 'and I will be right beside you.'),
        'instruct': ('A warm elderly man storybook voice speaking clear English. Calm, dependable, affectionate and quietly '
                     'humorous, with a subtle South African English character. Medium-low pitch, measured pace, natural age '
                     'texture and clear articulation. Strong but never stern, booming or theatrical. Clean studio recording '
                     'with no music, noise or reverberation.')
    },
}

(OUT_DIR / 'voice_specs.json').write_text(json.dumps(VOICE_SPECS, indent=2), encoding='utf-8')
print(json.dumps({k: v['text'] for k, v in VOICE_SPECS.items()}, indent=2))

In [ ]:
# @title 4. Generate three candidates per character
CANDIDATES_PER_CHARACTER = 3
BASE_SEED = 20260909
records = []

for speaker_index, (speaker, spec) in enumerate(VOICE_SPECS.items()):
    speaker_dir = OUT_DIR / speaker
    speaker_dir.mkdir(parents=True, exist_ok=True)
    for candidate in range(1, CANDIDATES_PER_CHARACTER + 1):
        seed = BASE_SEED + speaker_index * 100 + candidate
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        wavs, sample_rate = model.generate_voice_design(
            text=spec['text'],
            language='English',
            instruct=spec['instruct'],
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
        )
        output_path = speaker_dir / f'{speaker}_{candidate:02d}.wav'
        sf.write(output_path, wavs[0], sample_rate, subtype='PCM_16')
        duration = len(wavs[0]) / sample_rate
        records.append({
            'speaker': speaker, 'candidate': candidate, 'seed': seed,
            'sample_rate': sample_rate, 'duration_seconds': round(duration, 3),
            'path': str(output_path), 'text': spec['text'], 'instruct': spec['instruct'],
        })
        print(f'{speaker} candidate {candidate}: {duration:.2f}s -> {output_path}')

(OUT_DIR / 'generation_manifest.json').write_text(json.dumps(records, indent=2), encoding='utf-8')
print('Generated', len(records), 'candidate voices.')

In [ ]:
# @title 5. Package outputs
import shutil
archive = shutil.make_archive('/content/khaya_qwen3_voice_candidates', 'zip', OUT_DIR)
print('Archive:', archive, os.path.getsize(archive), 'bytes')
print('OUTPUT_ARCHIVE=/content/khaya_qwen3_voice_candidates.zip')

In [ ]:
# @title 6. Copy to Google Drive after Drive is mounted
from pathlib import Path
import shutil
drive_root = Path('/content/drive/MyDrive')
if not drive_root.exists():
    print('DRIVE_NOT_MOUNTED: mount Google Drive once, then rerun this cell.')
else:
    drive_dir = drive_root / 'vidio' / 'khaya-wants-to-row' / 'voices' / 'qwen3-voice-design-candidates'
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copytree(OUT_DIR, drive_dir, dirs_exist_ok=True)
    shutil.copy2('/content/khaya_qwen3_voice_candidates.zip', drive_dir.parent / 'khaya_qwen3_voice_candidates.zip')
    print('DRIVE_OUTPUT=', drive_dir)